In [1]:
from openai import OpenAI
from scraper import fetch_website_links, fetch_website_contents
from IPython.display import Markdown, display, update_display
import json

In [2]:
openai = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")

In [ ]:
fetch_website_links("https://huggingface.co/")

In [ ]:
display(Markdown(fetch_website_contents("https://huggingface.co/")))

In [3]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
display(Markdown(get_links_user_prompt("https://www.huggingface.co")))

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling gemma3:4b")
    response = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    try :
        print(f"Found {len(links['links'])} relevant links")
        return links
    except Exception as e:
        return links
    

In [ ]:
relevant_links=select_relevant_links("https://www.huggingface.co")

In [6]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
url = "https://edwarddonner.com"
contents = fetch_website_contents(url)
relevant_links = select_relevant_links(url)
result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
for link in relevant_links['links']:
    result += f"\n\n### Link: {link['type']}\n"
    result += fetch_website_contents(link["url"])

In [ ]:
result

In [ ]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

In [10]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [7]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [ ]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

In [ ]:
get_brochure_user_prompt("Edward Donner", "https://edwarddonner.com")

In [8]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [9]:
create_brochure("Edward Donner", "https://edwarddonner.com")

NameError: name 'brochure_system_prompt' is not defined

In [11]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemma3:4b",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [16]:
stream_brochure("Edward Donner", "https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemma3:4b
Found 7 relevant links


## Edward Donner: Shaping the Future of Work with AI

**A Visionary Leader in Generative AI & Talent Acquisition**

**Who We Are:**

Edward Donner is a pioneering figure in the world of Artificial Intelligence, driven by a passion for applying AI to solve real-world problems, particularly in the critical area of talent acquisition. He’s a founder, educator, and thought leader, dedicated to helping individuals find fulfilling roles and contributing to a more prosperous and engaged workforce.

**Our Core Focus: Nebula.io & Beyond**

Nebula.io, co-founded by Mr. Donner, is revolutionizing recruitment with its patented AI-powered platform.  It dramatically improves the speed and accuracy of matching candidates with suitable roles, eliminating the need for traditional keyword searches.  This innovative approach is powered by advanced generative AI and machine learning, aiming to ultimately improve human prosperity by helping individuals find their ideal career paths. 

**Key Areas of Expertise & Innovation:**

* **Generative AI in Talent Acquisition:** Utilizing advanced AI models to understand candidate profiles and identify optimal role matches.
* **Agentic AI:**  Applying the power of agents to navigate complex recruitment challenges and guide candidates toward their ideal careers.
* **Ikigai Driven Approach:**  Inspired by the Japanese concept of *Ikigai*, we focus on helping individuals find roles where they will be most fulfilled and successful.
* **AI Education & Empowerment:**  Mr. Donner is a prolific creator of top-rated Udemy courses, with over 500,000 enrolled across the globe.  His courses (available here: [link to full curriculum]) cover topics like:
    * Proficient AI Engineer
    * AI Coder: Vibe Coder to Agentic Engineer
    * AI Builder with n8n – Create Agents and Voice Agents
    * AI Engineering MLOps Track – Deploy AI to Production

**The Edward Donner Legacy:**

* **Previous Ventures:** Mr. Donner’s career is marked by innovation, starting with the AI startup, “untapt”, which was acquired in 2021.
* **Thought Leadership:**  A frequent speaker, he's delivered over 180,000 views on talks at ODSC in Boston and runs live events for O’Reilly and Pearson. 
* **The "Connect Four" Arena:**  Mr. Donner’s passion for AI is showcased in the engaging "Connect Four" arena, a battleground for LLMs, demonstrating the power of strategic AI.



**Connect with Edward Donner:**

* **Website:** [www.edwarddonner.com](http://www.edwarddonner.com)
* **LinkedIn:** [Follow Edward Donner on LinkedIn](Link to LinkedIn)
* **Twitter:** [Follow Edward Donner on Twitter](Link to Twitter)
* **Facebook:** [Follow Edward Donner on Facebook](Link to Facebook)

**Are you looking for a talented AI engineer or a team to help you transform your recruitment process?**  Reach out to Edward Donner today!

In [15]:
stream_brochure("Anthropic", "https://en.wikipedia.org/wiki/Anthropic")

Selecting relevant links for https://en.wikipedia.org/wiki/Anthropic by calling gemma3:4b


KeyError: 'links'

In [17]:
stream_brochure("Github", "https://github.com/")

Selecting relevant links for https://github.com/ by calling gemma3:4b
Found 14 relevant links


---

**GitHub: Empowering the Global Developer Community**

**For Prospective Customers, Investors, & Recruiters**

**What is GitHub?**

GitHub is the world’s leading platform for developers, providing the tools and services needed to build, collaborate, and secure software. It’s more than just a code repository; it’s the central hub for the global developer community. 

**Why Choose GitHub?**

* **The Largest Developer Community:** GitHub boasts over 100 million users globally, fostering collaboration and knowledge sharing.
* **AI-Powered Development:** GitHub is integrating AI through platforms like GitHub Copilot, significantly boosting developer productivity and code quality.
* **Secure and Reliable:**  GitHub offers comprehensive security features, including GitHub Advanced Security and Secret Protection, making it a trusted platform for managing sensitive code. 
* **Versatile Solutions:**  GitHub provides solutions addressing a broad range of needs, from App Modernization and DevSecOps to CI/CD and DevOps.


**Solutions for Your Needs:**

**For Customers:**

* **App Modernization:** Quickly update legacy applications using GitHub’s collaborative features and CI/CD pipelines.
* **DevSecOps:** Integrate security practices into every stage of the development lifecycle with GitHub Advanced Security and integrated tools.
* **DevOps & CI/CD:** Automate software delivery with robust CI/CD pipelines and integrations.
* **Industry-Specific Solutions:** Customized solutions for Healthcare, Financial Services, Manufacturing, Government and other sectors. 


**For Investors:**

* **Growth Opportunity:** The demand for software development is booming, and GitHub is positioned as a key player in this expanding market.
* **Innovation Leadership:** GitHub’s commitment to AI-powered development and open-source initiatives positions it as an innovator in the technology landscape.
* **Large & Active User Base:**  A massive and engaged user base represents a significant market opportunity. 
* **Expanding Revenue Streams:**  The company’s diverse offerings, including GitHub Advanced Security and Copilot for Business, provide multiple avenues for growth.


**For Recruiters:**

* **High Demand for Developers:**  GitHub offers opportunities to hire skilled developers who are already utilizing the world’s most popular development platform.
* **Modern Tech Stack:**  Working with GitHub exposes talent to cutting-edge technologies and development methodologies.
* **Dynamic & Collaborative Culture:**  GitHub's vibrant community and collaborative approach foster a stimulating and rewarding workplace. 
* **Variety of Roles:**  Opportunities exist across development, security, product, engineering, and more. 

**GitHub’s Core Values & Culture:**

GitHub’s culture is driven by a commitment to open source, collaboration, and innovation. They empower developers to contribute to the world's largest collection of software, and emphasize a community-based approach to problem-solving. 

**Resources:**

* **Website:** [https://github.com/](https://github.com/)
* **GitHub Sponsors:** [https://github.com/sponsors](https://github.com/sponsors) - Support open source developers!


---
